# WLASL chase v4 (Colab T4)

Beat **69.2%** and aim for **85%** on the 50-gloss val set (117 clips).

New vs v3: hand-bone geometry, mirror/time-warp on weak glosses, CNN-BiLSTM, kNN, and a frozen v3 voter.

**Runtime → T4 GPU** before running. Download `outputs/chase_colab_v4_results.zip` when finished.


In [ ]:
# @title 1) Setup paths
from pathlib import Path
ZIP_PATH = Path('/content/wlasl_colab_chase_v4.zip')
# ZIP_PATH = Path('/content/drive/MyDrive/wlasl_colab_chase_v4.zip')
USE_DRIVE = False
DRIVE_OUT = Path('/content/drive/MyDrive/wlasl_chase_outputs')
WORK = Path('/content/wlasl_colab_chase_v4')
OUT = WORK / 'outputs' / 'chase_colab_v4'
print('ZIP exists:', ZIP_PATH.exists(), ZIP_PATH)

In [ ]:
# @title 2) Optional Drive
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# @title 3) Unzip
import zipfile
from pathlib import Path
CONTENT = Path('/content')
WORK = CONTENT / 'wlasl_colab_chase_v4'
meta = WORK / 'data' / 'processed' / 'wlasl_landmarks_v4' / 'metadata.json'
if not meta.exists():
    assert ZIP_PATH.exists(), f'Upload zip first: {ZIP_PATH}'
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(CONTENT)
assert meta.exists(), meta
OUT = WORK / 'outputs' / 'chase_colab_v4'
OUT.mkdir(parents=True, exist_ok=True)
print('landmarks', meta)
print('v3 expert', (WORK / 'models' / 'experts' / 'v3' / 'wlasl_landmark_ensemble.pt').exists())

In [ ]:
# @title 4) Deps + GPU
import subprocess, sys, torch
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'joblib', 'scikit-learn', 'numpy'])
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
else:
    print('WARNING: set Runtime to T4 GPU')

In [ ]:
# @title 5) Train chase v4
import os, sys, json, time
os.chdir(WORK)
sys.path.insert(0, str(WORK))
from modules.recognition.wlasl_chase_v4 import train_chase_v4

# If you hit a CUDA/RAM OOM, rerun with aug_copies=2 and n_transformer_seeds=2.
t0 = time.time()
report = train_chase_v4(
    processed_dir=str(WORK / 'data' / 'processed' / 'wlasl_landmarks_v4'),
    output_dir=str(OUT),
    expert_dir=str(WORK / 'models' / 'experts' / 'v3'),
    epochs=100,
    n_transformer_seeds=3,
    n_tgcn_seeds=2,
    aug_copies=4,
)
print(f'Done in {(time.time()-t0)/60:.1f} min')
print(json.dumps({
    'val_accuracy': report['val_accuracy'],
    'baseline_v3': 0.6923,
    'target': 0.85,
    'accuracy_met_85': report['accuracy_met'],
    'beat_v3': report['beat_v3'],
    'blend_weights': report['blend_weights'],
    'transformer': report['transformer_ensemble_val_accuracy'],
    'tgcn': report['tgcn_val_accuracy'],
    'cnn': report['cnn_val_accuracy'],
    'hgb': report['hgb_val_accuracy'],
    'knn': report['knn_val_accuracy'],
}, indent=2))

In [ ]:
# @title 6) Package results
import json, shutil, zipfile
from pathlib import Path
summary = {
    'run': 'chase_colab_v4',
    'val_accuracy': report['val_accuracy'],
    'accuracy_met_85': report['accuracy_met'],
    'beat_v3': report['beat_v3'],
    'baseline_v3': 0.6923076923076923,
    'blend_weights': report['blend_weights'],
    'n_val': report['n_val'],
    'n_train': report['n_train'],
    'transformer_ensemble_val_accuracy': report['transformer_ensemble_val_accuracy'],
    'tgcn_val_accuracy': report['tgcn_val_accuracy'],
    'cnn_val_accuracy': report['cnn_val_accuracy'],
    'hgb_val_accuracy': report['hgb_val_accuracy'],
    'knn_val_accuracy': report['knn_val_accuracy'],
    'per_class_recall': report['per_class_recall'],
    'source': 'colab_t4_chase_v4',
}
(OUT / 'colab_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
result_zip = WORK / 'outputs' / 'chase_colab_v4_results.zip'
with zipfile.ZipFile(result_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT.rglob('*')):
        if p.is_file():
            zf.write(p, arcname=str(Path('chase_colab_v4') / p.relative_to(OUT)))
print('Wrote', result_zip, 'size_mb', round(result_zip.stat().st_size/1e6, 2))
if USE_DRIVE:
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    shutil.copy2(result_zip, DRIVE_OUT / 'chase_colab_v4_results.zip')
try:
    from google.colab import files
    files.download(str(result_zip))
except Exception as e:
    print('Auto-download skipped:', e, result_zip)